# Vaje 2: Linearni napovedni modeli

Na današnjih vajah se bomo posvetili linearnim napovednim modelom, ki so en izmed najbolj pomembnih stebrov strojnega (in statističnega) učenja. Ti modeli se še vedno uporabljajo v bančništvu, zavarovalništvu in so pogosto prvi model, ki ga podatkovni znanstvenik uporabi, ko začne reševati določen problem.

**Zakaj so linearni napovedni modeli pomembni?**
- Interpretabilnost: Za razliko od modelov "črne škatle" (angl. black-box), nam linearni modeli natančno povedo, kako vsaka vhodna spremenljivka vpliva na končno napoved. V financah ali medicini je razumevanje zakaj je model prišel do določene odločitve pogosto pomembnejše od same natančnosti.
- Učinkovitost in razširjenost: So računsko izjemno lahki, hitri za učenje in služijo kot odličen "baseline" (izhodiščni model), s katerim primerjamo vse kompleksnejše pristope.
- Matematična čistost: Modeli temeljijo na trdnih temeljih linearne algebre in optimizacije in so tesno povezani s funkcijami napake, ki jih pogosto uporabimo za ocenjevanje točnosti modela.

Pogledali si bomo dva modela: linearno regresijo, ki se uporablja za napovedovanje numeričnih ciljnih vrednosti (regresijo) in logistično regresijo, ki se uporablja za napovedovanje diskretnih ciljnih vrednosti (klasifikacijo).

## Naloga 1: Linearna regresija

Linearna regresija napove ciljno spremenljivko kot uteženo vsoto značilk, torej $\hat{y} = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \dots + \beta_p x_p$, kjer je $y$ napoved, $\beta_0$ začetna vrednost (intercept), $\beta_1, \dots, \beta_p$ uteži (koeficienti) in $x_1,\dots, x_p$ vrednosti značilk.

V teoriji uteži (koeficientov) linearnih regresije izračunamo s formulo $\hat{\beta} = (X^T X)^{-1} X^T y$, kjer je $X$ matrika s podatki in $y$ vektor ciljnih vrednosti. V praksi je računanje uteži z iterativno formulo (gradientni spust), saj je to računsko in prostorsko bolj učinkovito pri velikem številu podatkov in numerično bolj stabilno.

**Interpretacija uteži (koeficientov) linearne regresije**

V linearnem modelu vsaka utež $\beta_i$ pove, kako se spremeni ciljna spremenljivka, ko se ustrezna značilka spremeni za enoto. Interpretacija je odvisna od tipa značilke:
- Numerične značilke: Povečanje značilke za eno enoto poveča (ali zmanjša) napoved za vrednost uteži $\beta_i$.
- Binarne značilke (0/1): Razlika v napovedi med referenčno vrednostjo (0) in drugo vrednostjo (1) je enaka uteži $\beta_i$.
- Kategorične značilke (z več kategorijami): Uporabimo "one-hot encoding". Za $L$ kategorij potrebujemo $L-1$ stolpcev. Vsaka utež predstavlja razliko glede na referenčno kategorijo, ki je v modelu predstavljena implicitno (ko so vse ostale kategorije enake 0).
- Začetna vrednost (Intercept - $\beta_0$): Predstavlja napoved modela, ko so vse značilke enake 0 (oziroma so v referenčni kategoriji).

_Opomba: Če so podatki standardizirani (povprečje 0, standardni odklon 1), presečišče predstavlja napovedano vrednost pri povprečnih vrednostih vseh značilk._

**Predpostavke in lastnosti linearnih napovednih modelov**

Da bi bil linearni model (linearna regresija) veljaven in učinkovit, mora zadoščati določenim teoretičnim predpostavkam:
- Linearnost: Model predpostavlja, da je ciljna spremenljivka linearna kombinacija vhodnih značilk. To omogoča enostavno interpretacijo (učinki so aditivni), vendar nas omejuje pri modeliranju kompleksnih, nelinearnih povezav (razen če vključimo interakcijske člene).
- Normalnost napak: Predpostavljamo, da napake (reziduali) sledijo normalni porazdelitvi. To je ključno za veljavnost statističnih testov in intervalov zaupanja.
- Homoskedastičnost (konstantna varianca): Varianca napak mora biti enaka čez celoten prostor značilk. V praksi je to pogosto kršeno (npr. pri dražjih hišah so napake v ceni običajno večje kot pri poceni hišah), kar pomeni, da so napovedi pri ekstremnih vrednostih manj zanesljive.
Linearity
- Neodvisnost: Vsak opazovan podatek je neodvisen od ostalih. Če imamo opravka s časovnimi serijami ali ponovljenimi meritvami na istih subjektih, ta predpostavka odpade in potrebujemo naprednejše modele
- Fiksne značilke: Predpostavljamo, da so vhodni podatki podani brez napak pri merjenju. Čeprav to v praksi redko drži, bi modeliranje napak v samih vhodnih podatkih zahtevalo kompleksne modele, ki se jim v osnovni regresiji izognemo.
- Odsotnost multikolinearnosti: Značilke ne smejo biti med seboj močno korelirane. Če sta dve značilki visoko korellirani, postane ocenjevanje uteži (koeficientov) nestabilno, saj model ne more ločiti, katera značilka dejansko povzroča učinek.

Ustvarimo najprej podatkovno množico z vsaj desetimi spremenljivkami, na kateri bomo testiral pristop. Funkcija ciljne spremenljivke, ki jo definiraš, naj bo numerična, naj vsebuje šum in ima največ 5 spremenljivk.

Pomagamo si lahko s [funkcijo numpy.random.random_sample()](https://numpy.org/doc/1.25/reference/random/generated/numpy.random.random_sample.html) in s [funkcijo numpy.random.normal()](https://numpy.org/doc/1.25/reference/random/generated/numpy.random.normal.html), ki ustvari matriko z naključnimi podatki.

In [6]:
import numpy as np

In [7]:
# Z numpy.random.random() zgeneriramo matriko velikosti 1000x10 (1000 vrstic, 10 stolpcev)
x = np.random.random((1000, 10))
# Definiramo ciljno spremenljivko in ji dodamo normalno porazdeljen šum z numpy.random.normal(pričakovana vrednost, standardna deviacija)
# Z x[:, i] izberemo celoten i-ti stolpec matrike
y = x[:, 0] * x[:, 1] - 3 * x[:, 2] +  6*x[:, 3] - 3 * x[:, 4] * x[:, 3] * x[:, 2] + np.random.normal(0, 0.2, size=1000)
# Definiramo masko, ki nam pove katere spremenljivke definirajo ciljno spremenljivko
y_mask = [True, True, True, True, True, False, False, False, False, False]

Sedaj bomo podatke razdelili na učno in testno množico, pri čemer bo učna množica vsebovala 80% podatkov, testna množica pa 20%.

**Zakaj delimo podatke na učno in testno množico?**

Ko gradimo model strojnega učenja, naš cilj ni le, da se model dobro prilagodi podatkom, ki jih že imamo, temveč da natančno napoveduje vrednosti na novih, še nevidenih podatkih.

Če bi model preverjali na istih podatkih, na katerih smo ga učili, bi dobili napačen občutek o njegovi kakovosti. Prišlo bi do pojava, ki mu pravimo preveliko prilagajanje (angl. overfitting).
Glavni razlogi za delitev (80:20):

1. Preprečevanje prevelikega prilagajanja (Overfitting): Model z velikim številom parametrov (npr. polinomska regresija visoke stopnje) lahko "zadene" vse točke v učni množici, vključno s šumom in naključnimi odstopanji. Tak model bo imel na učni množici napako 0, na novih podatkih pa bo popolnoma odpovedal. Testna množica nam služi kot "realnostni test".

2. Ocena zmožnosti posploševanja: Testna množica predstavlja "nevidne" podatke iz prihodnosti. Z njo ocenimo, kako robusten je model v realnem svetu. Če je napaka na testni množici bistveno večja od napake na učni množici, vemo, da model ni uporaben.

3. Nepristranska evalvacija: V strojnem učenju testna množica služi kot neodvisen sodnik. Pravilo je: modela nikoli ne učimo na testnih podatkih. Če bi to storili, bi informacije iz testne množice "ušle" v model (angl. data leakage), rezultat pa bi bila nerealno visoka natančnost.

Podatke lahko razdelimo na dva načina: ročno ali pa s pomočjo knjižnice [scikit-learn](https://scikit-learn.org/stable/index.html). Ta knjižnica je uporabna, saj vsebuje veliko zbirko optimiziranih modelov. Ti modeli si delijo vmesnik, ki je preprost za uporabo. Poleg tega knjižnica vsebuje veliko funkcij za predprocesiranje podatkov, evalvacijo modelov, računanje napak, ...

In [3]:
!pip install scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 10.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 10.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [scikit-learn] [scikit-learn]


In [4]:
from sklearn.model_selection import train_test_split

In [8]:
# Ročno
# Izberemo prvih (4/5)*1000=800 vrstic, ki bodo sestavljale učno množico
x_train = x[:800, :]
y_train = y[:800]
# Izberemo zadnjih (1/5)*1000=200 vrstic, ki bodo sestavljale testno množico
x_test = x[800:, :]
y_test = y[800:]

# S pomočjo sklearn.model_selection.train_test_split
#
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2)

Poglejmo si torej kako s knjižnico sklearn definiramo in natreniramo model [linearne regresije (sklearn.linear_model.LinearRegression)](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html). Za računanje napake bomo uporabili RMSE (root mean squared error), ki je definiran kot $\text{RMSE}(y, \hat{y}) = \sqrt{\frac{1}{n}\sum_{i=1}^n(y_i - \hat{y}_i)^2}$$. Tega lahko v knjižnici sklearn najdemo pod [sklearn.metrics.root_mean_squared_error()](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.root_mean_squared_error.html).

In [10]:
from sklearn.metrics import root_mean_squared_error
from sklearn.linear_model import LinearRegression

In [11]:
# 1. Definicija sklearn modela, v našem primeru linearne regresije
lm = LinearRegression()

# 2. Treniranje sklearn modela s funkcijo fit(učni podatki, ciljna vrednost učnih podatkov)
lm.fit(x_train, y_train)

# 3. S funkcijo .predict(testni podatki) napovemo vrednost ciljne spremenljivke za testne podatke
y_pred = lm.predict(x_test)

# 4. Izračunamo napako med napovedmi in pravo vrednostjo ciljne spremenljivke in jo izpišemo
print(f"RMSE error on test data: {root_mean_squared_error(y_test, y_pred)}")

RMSE error on test data: 0.2970724312993593


1.a: Poglej [dokumentacijo za linearno regresijo](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html) in s pomočjo ustreznih metod in atributov odgovori na naslednja vprašanja:
- Kateri koeficienti spremenljivk pozitivno korelirajo s ciljno spremenljivko in kateri negativno?
- Katera spremenljivka najbolj vpliva na napoved modela?
- Kako bi brez klica metode LinearRegression.predict() ugotovil vrednost, ki bi jo model napovedal za vektor ničel in kakšna je ta vrednost?
- Kolikšna je razlika med napako na učni in testni množici?
- Koliko na točnost vplivajo spremenljivke, ki se v ciljni funkciji ne pojavijo?

Odgovori:
- ...

## Naloga 2: Logistična regresija

Linearna regresija ima neomejeno zalogo vrednosti ($-\infty$, $\infty$), zato za klasifikacijo ni primerna, saj vrednosti nad 1 in pod 0 nimajo smisla. Za klasifikacijo je zato bolj primerna logistična regresija, ki napoved omeji na interval (0,1). V logistični regresiji vrednost ciljne spremenljivke napovemo s formulo $\hat{y} = \frac{1}{1+e^{-(\beta_0 + \beta_1 x_1 + \beta_2 x_2 + \ldots + \beta_n x_n)}}$. Če je vrednost $\hat{y}$ višja od 0.5, napovemo 1, čene 0. V osnovi logistična regresija nima zaprte formule za direkten izračun uteži, zato njene uteži računamo iterativno s pomočjo gradientnega spusta.

**Interpretacija uteži v logistični regresiji**

V logistični regresiji uteži ne vplivajo na verjetnost linearno, temveč **multiplikativno na razmerje obetov** (angl. *odds ratio*). Ker model napoveduje vrednosti med 0 in 1, moramo za linearno interpretacijo enačbo preoblikovati v obliko **logaritma obetov** (angl. *log-odds* ali *logit*):

$$\ln\left(\frac{p}{1-p}\right) = \beta_0 + \beta_1 x_1 + \dots + \beta_n x_n$$

- Obeti (Odds): Razmerje med verjetnostjo, da se dogodek zgodi ($p$), in verjetnostjo, da se ne zgodi ($1-p$).
- Log-odds: Linearna kombinacija uteži. Povečanje $x_i$ za enoto poveča log-odds za vrednost $\beta_i$.
- Razmerje obetov (Odds Ratio - $e^{\beta}$): Ker je razmišljanje v logaritmih težko, večina interpretira eksponent uteži. Faktor $e^{\beta_i}$ nam pove, za koliko se **pomnožijo** obeti pri spremembi $x_i$ za eno enoto.

**Interpretacija glede na tip značilke:**
- Numerična: Povečanje $x_i$ za 1 enoto spremeni obete za faktor **$e^{\beta_i}$**.
- Binarna (0/1): Sprememba iz referenčne kategorije (0) v drugo (1) spremeni obete za faktor **$e^{\beta_i}$**.
- Kategorična ($L$ kategorij): Uporabimo *one-hot encoding* ($L-1$ stolpcev). Vsaka utež predstavlja multiplikativno spremembo obetov glede na referenčno kategorijo.
- Presečišče ($\beta_0$): Ko so vse numerične vrednosti 0 in kategorične v referenčnih stanjih, so ocenjeni obeti enaki **$e^{\beta_0}$**. |

**Prednosti in omejitve:**
- Logistična regresija podeduje veliko prednosti/lastnosti linearne regresije
- Napovedovanje verjetnosti: Model ne poda le končne klasifikacije (0 ali 1), temveč izračuna verjetnost pripadnosti razredu. Velika razlika je, ali model napove razred z 51-odstotno ali 99-odstotno gotovostjo, kar je ključno pri obvladovanju tveganj v bančništvu.
- Kalibracija: Če je model dobro umerjen, napovedana verjetnost 0,6 dejansko pomeni, da se bo dogodek v povprečju zgodil v 60 % primerov.
- Omejena izrazna moč: Podobno kot pri linearni regresiji, model težko zajame nelinearne odnose ali interakcije med značilkami, razen če jih v model vključimo ročno (npr. s produkti značilk).
- Multiplikativna interpretacija: Zaradi eksponentne narave uteži je interpretacija težja in manj intuitivna kot aditivna interpretacija pri linearni regresiji.
- Problem popolne ločljivosti (Complete Separation): Če obstaja značilka, ki popolnoma loči oba razreda (npr. vsi z višino nad 180 cm so moški), model ne more konvergirati. Matematično: Optimalna utež bi v tem primeru težila v neskončnost, kar povzroči numerično nestabilnost. Rešitev: Problem rešujemo z regularizacijo (kaznovanjem velikih uteži) ali z uvedbo apriornih porazdelitev uteži.

**N-arna klasifikacija (Multi-class classification):**

Obstajajo različne strategije kako logistično regresijo razširimo iz binarne klasifikacije (kjer imamo le dva razreda) na n-arno klasifikacijo.
- En proti vsem (One-Against-All): Za vsak razred naučimo en model logistične regresije, kjer napovedujemo ta razred. Razred napovemo tako, da poženemo vse modele in izberemo razred, ki ima najvišjo vrednost
- En proti enemu (One-Against-One): Za vsak par razredov naredimo en model logistične regresije. Napoved naredimo tako, da poženemo vse modele, preštejemo število "zmag" vsakega razreda in napovemo tistega, ki jih ima največ.

Sestavimo najprej podatke, ki jih bomo uporabljali za klasifikacijo in jih razdelimo na učno in testno množico.

In [ ]:
# Z numpy.random.random() zgeneriramo matriko velikosti 1000x10 (1000 vrstic, 10 stolpcev)
x = np.random.random((1000, 10))
# Definiramo numerično ciljno spremenljivko in ji dodamo normalno porazdeljen šum z numpy.random.normal(pričakovana vrednost, standardna deviacija)
# Z x[:, i] izberemo celoten i-ti stolpec matrike
y_continuous = 1 + x[:, 0] * x[:, 1] - 3 * x[:, 2] +  6*x[:, 3] - 3 * x[:, 4] * x[:, 3] * x[:, 2] + np.random.normal(0, 0.2, size=1000)
# Spremenimo numerične vrednosti v diskretno ciljno vrednost. Recimo, naj bo ciljna vrednost 0, če je vrednost pod povprečjem in 1 čene
y_discrete = y_continuous > np.mean(y_continuous)
# Definiramo masko, ki nam pove katere katere spremenljivke definirajo ciljno spremenljivko
y_mask = [True, True, True, True, True, False, False, False, False, False]

# Množico razdelimo na učno in testno
x_train, x_test, y_train, y_test = train_test_split(x, y_discrete, test_size=0.2)

2.a: Natreniraj model logistične regresije in oceni njegovo točnost na testni množici z metriko accuracy. Uporabiš lahko [napovedni model sklearn.linear_model.LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) in [sklearn.metrics.accuracy_score()](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html).

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression

2.b: Odgovori na naslednja vprašanja:
- Kateri koeficienti spremenljivk pozitivno korelirajo s ciljno spremenljivko in kateri negativno?
- Katera spremenljivka najbolj vpliva na napoved modela?
- Kako bi brez klica metode LinearRegression.predict() ugotovil vrednost, ki bi jo model napovedal za vektor ničel?
- Kolikšna je razlika med napako na učni in testni množici?
- Koliko na točnost vplivajo spremenljivke, ki se v ciljni funkciji ne pojavijo?

## Naloga 3: K-najbližjih sosedov

Metoda kNN je eden najpreprostejših, a hkrati najučinkovitejših algoritmov za klasifikacijo in regresijo. Za razliko od linearne ali logistične regresije, kjer med učenjem iščemo optimalne uteži, se kNN "ne uči" v klasičnem smislu. Namesto tega si preprosto zapomni celotno učno množico.

Algoritem deluje na predpostavki, da imajo podobni podatki podobne izhode. Ko želimo napovedati razred za nov podatek:
1. Izračunamo razdaljo med novim podatkom in vsemi ostalimi podatki v učni množici.
2. Izberemo k podatkov, ki so mu najbližje.
3. Klasifikacija: Napovemo razred, ki je najpogostejši med temi k sosedi (večinsko glasovanje). Regresija: Izračunamo povprečje vrednosti teh k sosedov.

**Ključne značilnosti**
- Leno učenje (Lazy Learning): Model nima faze učenja, ki bi zahtevala računanje parametrov. Vsa računska zahtevnost se zgodi v fazi napovedovanja.
- Neparametrični model: Ne predpostavljamo nobene specifične oblike podatkov (npr. linearnosti). To omogoča modelu, da se prilagodi zelo kompleksnim oblikam meja med razredi.
- Parameter k (število najbližjih sosedov): ko je k majhen (npr. k=1) se osredotočimo le na najbližji podatek, kar lahko vodi do prevelikega prilagajanja (model sledi šumu). Pri velikem k-ju je odločitvena meja bolj gladka, a lahko spregledamo lokalne vzorce.
- Izbira razdalje: Najpogosteje uporabljamo evklidsko razdaljo, vendar lahko v posebnih prostorih uporabimo tudi manhattansko, čibiševo ali kosinusno razdaljo.
- Prekletstvo dimenzionalnosti: V prostorih z veliko značilkami postanejo razdalje med točkami "skoraj enake", kar zmanjša učinkovitost algoritma.
- Občutljivost na merilo: Ker algoritem temelji na razdaljah, je normalizacija podatkov (npr. pretvorba na interval [0,1]) nujna, sicer bodo značilke z večjimi vrednostmi prevladale. (poglej dodatno nalogo iz vaj 1)

V paketu sklearn se za klasifikacijo uporablja [razred sklearn.neighbors.KNeighborsClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html) za regresijo pa [razred sklearn.neighbors.KNeighborsRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsRegressor.html).

3.a: Primerjaj točnost linearne regresije s točnostjo modela k-najbližjih sosedov pri napovedovanju numeričnih spremenljivk.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsRegressor

In [ ]:
x = np.random.random((1000, 10))
# Definiramo ciljno spremenljivko in ji dodamo normalno porazdeljen šum z numpy.random.normal(pričakovana vrednost, standardna deviacija)
# Z x[:, i] izberemo celoten i-ti stolpec matrike
y = x[:, 0] * x[:, 1] - 3 * x[:, 2] +  6*x[:, 3] - 3 * x[:, 4] * x[:, 3] * x[:, 2] + np.random.normal(0, 0.2, size=1000)

3.b: Primerjaj točnost logistične regresije s točnostjo modela k-najbližjih sosedov pri napovedovanju diskretnih spremenljivk. Uporabi kodo iz naloge 2.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
x = np.random.random((1000, 10))
# Definiramo numerično ciljno spremenljivko in ji dodamo normalno porazdeljen šum z numpy.random.normal(pričakovana vrednost, standardna deviacija)
# Z x[:, i] izberemo celoten i-ti stolpec matrike
y_continuous = 1 + x[:, 0] * x[:, 1] - 3 * x[:, 2] +  6*x[:, 3] - 3 * x[:, 4] * x[:, 3] * x[:, 2] + np.random.normal(0, 0.2, size=1000)
# Spremenimo numerične vrednosti v diskretno ciljno vrednost. Recimo, naj bo ciljna vrednost 0, če je vrednost pod povprečjem in 1 čene
y_discrete = y_continuous > np.mean(y_continuous)


3.c: Poišči dva primera in ju nariši s pomočjo scatter plota:
  - Primer, v katerem bo KNN model deloval dobro in linearna regresija slabo
  - Primer, v katerem bo linearna regresija delovala dobro in KNN model slabo

_Namig: Graf lahko narišeš tako, da bo imel podatek na x osi vrednost spremenljivke, na y osi pa vrednost napovedi, ki jo dobiš z natreniranim modelom. Če metodo plt.scatter kličeš večkrat brez klica metode plt.show(), bo imel vsak klic plt.scatter svojo barvo točk. S parametrom s, lahko v funkciji plt.scatter določiš velikost točk na grafu._

Odgovor za primer 1:

Odgovor za primer 2: